# MIND-small: impression counts and entity confidence features

Derives features from the preprocessed handoff (`data/processed/<version>`) for every partition listed in its manifest.

| Feature | One row per | Definition |
|---|---|---|
| `history_length` | `event_key` | Number of previously clicked articles (repeated IDs count each time) |
| `num_candidates` | `event_key` | Number of candidate articles in the impression |
| `num_embedded_entities` | `news_id` | Distinct title/abstract entities that have a pretrained embedding |
| `entity_conf_mean`, `entity_conf_min` | `news_id` | Mean / minimum linking confidence of those entities (0 if none) |
| `ordinary`, `weighted` | `news_id` | 100-d entity vector: plain vs confidence-weighted average of those entities' embeddings |

Impression counts use the full candidate lists (before negative sampling), so they can be joined to sampled training rows by `event_key`.

## 1. Setup

Finds the repository root (or set `BT4222_PROJECT_ROOT`). Defaults to `data/processed/v1`; set `BT4222_PROCESSED_DIR` to use another version, e.g. one on Drive that includes the test partition. Entity embeddings are read from the raw folders (`MIND_RAW_DIR`, default `data/raw`).

In [1]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# In Colab, clone/open the repository first; set BT4222_PROJECT_ROOT if needed.
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount('/content/drive')

configured_root = os.environ.get('BT4222_PROJECT_ROOT')
if configured_root:
    PROJECT_ROOT = Path(configured_root).expanduser().resolve()
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / 'src/data_loading.py').is_file()), None)
if PROJECT_ROOT is None or not (PROJECT_ROOT / 'src/data_loading.py').is_file():
    raise FileNotFoundError('Open the cloned repo, or set BT4222_PROJECT_ROOT to its path.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import load_partition

# Environment variables override the default folders (e.g. Drive paths).
PROCESSED_DIR = Path(os.environ.get('BT4222_PROCESSED_DIR', PROJECT_ROOT / 'data/processed/v1')).expanduser()
RAW_DIR = Path(os.environ.get('MIND_RAW_DIR', PROJECT_ROOT / 'data/raw')).expanduser()
FEATURES_DIR = Path(os.environ.get('BT4222_FEATURES_DIR', PROJECT_ROOT / 'outputs/features')).expanduser()
manifest = json.loads((PROCESSED_DIR / 'manifest.json').read_text(encoding='utf-8'))
PARTITIONS = manifest['partitions']
IMPRESSION_OUTPUT_DIR = FEATURES_DIR / 'impression_counts' / manifest['dataset_version']
ENTITY_OUTPUT_DIR = FEATURES_DIR / 'entity_confidence' / manifest['dataset_version']

print('Processed data:', PROCESSED_DIR)
print('Raw data:      ', RAW_DIR)
print('Partitions:    ', ', '.join(PARTITIONS))
print('Feature output:', FEATURES_DIR)

Processed data: /Users/roniheikkonen/Desktop/NUS/Y3S1/BT4222/bt4222/data/processed/v1
Raw data:       /Users/roniheikkonen/Desktop/NUS/Y3S1/BT4222/bt4222/data/raw
Partitions:     train, validation
Feature output: /Users/roniheikkonen/Desktop/NUS/Y3S1/BT4222/bt4222/outputs/features


## 2. Load the partitions

`load_partition` restores string IDs, timestamps and the parsed history/entity lists.

In [2]:
data = {partition: load_partition(partition, PROCESSED_DIR) for partition in PARTITIONS}
for partition, (events, candidates, news) in data.items():
    print(f'{partition}: {len(events):,} impressions, {len(candidates):,} candidates, {len(news):,} articles')

train: 95,071 impressions, 3,389,342 candidates, 43,370 articles
validation: 61,894 impressions, 2,454,102 candidates, 38,421 articles


## 3. Impression features

`history_length` is recomputed from `history_ids` and checked against the column preprocessing already exports. `num_candidates` counts each impression's rows in `candidates`.

In [3]:
IMPRESSION_FEATURES = ['history_length', 'num_candidates']


def derive_impression_features(events, candidates):
    """One row per event_key (in events order) with history_length and num_candidates."""
    features = events[['event_key']].copy()
    features['history_length'] = events['history_ids'].str.len().astype('int32')
    assert features['history_length'].eq(events['history_length']).all(), \
        'history_length disagrees with the preprocessed column'

    counts = (candidates.groupby('event_key', sort=False).size()
              .astype('int32').reset_index(name='num_candidates'))
    features = features.merge(counts, on='event_key', how='left', validate='one_to_one')
    assert features['num_candidates'].notna().all(), 'Some impressions have no candidates'
    return features

In [4]:
split_summary = pd.read_csv(PROCESSED_DIR / 'split_summary.csv', index_col='partition')
impression_features_by_partition = {}

for partition, (events, candidates, _) in data.items():
    features = derive_impression_features(events, candidates)

    # Totals must match what preprocessing recorded for this partition.
    expected = split_summary.loc[partition]
    assert len(features) == expected['events']
    assert features['num_candidates'].sum() == expected['candidate_exposures']
    impression_features_by_partition[partition] = features

display(impression_features_by_partition['train'].head())

,event_key,history_length,num_candidates
0,MINDsmall_train/behaviors.tsv:20112,15,31
1,MINDsmall_train/behaviors.tsv:13807,93,29
2,MINDsmall_train/behaviors.tsv:27660,7,5
3,MINDsmall_train/behaviors.tsv:152217,8,56
4,MINDsmall_train/behaviors.tsv:42166,4,48


### Summary statistics

**Insight:** Both features are right-skewed (mean above median, long upper tails), so a `log1p` transform or clipping may help scale-sensitive models. Distributions are broadly similar across partitions.

In [5]:
summary = pd.concat({
    partition: features[IMPRESSION_FEATURES].describe(percentiles=[0.25, 0.5, 0.75, 0.99]).T
    for partition, features in impression_features_by_partition.items()
})
display(summary.drop(columns='count').round(2))

mean    std  min   25%   50%   75%    99%    max
train      history_length  32.05  39.45  0.0   8.0  19.0  41.0  188.0  558.0
           num_candidates  35.65  37.66  2.0   9.0  23.0  49.0  156.0  297.0
validation history_length  33.29  41.48  0.0   8.0  20.0  43.0  199.0  558.0
           num_candidates  39.65  39.86  2.0  11.0  26.0  54.0  182.0  299.0

## 4. Entity confidence features

Each title/abstract entity annotation carries the entity linker's `Confidence`. Per article, each distinct entity (Wikidata ID) is used once with its highest confidence, keeping only entities that have a pretrained embedding. Train/validation use the `MINDsmall_train` embeddings and test uses `MINDsmall_dev`, matching preprocessing.

- **Ordinary average:** mean of the article's entity vectors.
- **Confidence-weighted average:** $\sum_i c_i e_i \,/\, \sum_i c_i$, so less confident entities count less.

Articles without entity vectors get zero vectors and 0 for the confidence features. Everything is computed per article without labels, so nothing is fitted.

In [6]:
ENTITY_FIELDS = ['title', 'abstract']  # Use ['title'] for title entities only.
ENTITY_FEATURES = ['num_embedded_entities', 'entity_conf_mean', 'entity_conf_min']


def load_entity_embeddings(partition):
    source = 'MINDsmall_dev' if partition == 'test' else 'MINDsmall_train'
    vectors = pd.read_csv(RAW_DIR / source / 'entity_embedding.vec', sep='\t', header=None, index_col=0)
    return vectors.dropna(axis=1, how='all').astype('float32')  # Trailing tabs add an empty column.


def derive_entity_features(news, embeddings):
    """Per article (in news order): confidence features plus ordinary and weighted entity vectors."""
    entities = pd.DataFrame(
        [(news_id, entity['WikidataId'], entity['Confidence'])
         for field in ENTITY_FIELDS
         for news_id, annotations in zip(news['news_id'], news[f'{field}_entities_parsed'])
         for entity in annotations],
        columns=['news_id', 'wikidata_id', 'confidence'],
    )
    # One row per distinct entity in an article (highest confidence), only if it has an embedding.
    entities = (entities[entities['wikidata_id'].isin(embeddings.index)]
                .groupby(['news_id', 'wikidata_id'], sort=False)['confidence'].max().reset_index())

    features = (entities.groupby('news_id')['confidence']
                .agg(num_embedded_entities='size', entity_conf_mean='mean', entity_conf_min='min')
                .reindex(news['news_id'], fill_value=0).reset_index()
                .astype({'num_embedded_entities': 'int32', 'entity_conf_mean': 'float32',
                         'entity_conf_min': 'float32'}))

    vectors = embeddings.loc[entities['wikidata_id']].set_axis(entities.index)
    weights, article = entities['confidence'], entities['news_id']
    ordinary = vectors.groupby(article).mean()
    weighted = vectors.mul(weights, axis=0).groupby(article).sum().div(weights.groupby(article).sum(), axis=0)
    # Rows follow news order; articles without entity vectors stay all-zero.
    ordinary = ordinary.reindex(news['news_id'], fill_value=0).to_numpy('float32')
    weighted = weighted.reindex(news['news_id'], fill_value=0).to_numpy('float32')
    return features, ordinary, weighted

In [7]:
entity_features_by_partition, entity_vectors_by_partition = {}, {}

for partition, (_, _, news) in data.items():
    features, ordinary, weighted = derive_entity_features(news, load_entity_embeddings(partition))
    entity_features_by_partition[partition] = features
    entity_vectors_by_partition[partition] = {'ordinary': ordinary, 'weighted': weighted}
    print(f'{partition}: {features["num_embedded_entities"].gt(0).mean():.1%} of '
          f'{len(features):,} articles have entity vectors')

# Confidence features among articles with entities (the rest are 0).
summary = pd.concat({
    partition: features.loc[features['num_embedded_entities'] > 0, ENTITY_FEATURES]
    .describe(percentiles=[0.01, 0.1, 0.5]).T
    for partition, features in entity_features_by_partition.items()
})
display(summary.round(4))

train: 86.2% of 43,370 articles have entity vectors
validation: 85.4% of 38,421 articles have entity vectors


count    mean     std    min      1%  \
train      num_embedded_entities  37371.0  2.6746  1.6787  1.000  1.0000   
           entity_conf_mean       37371.0  0.9928  0.0130  0.901  0.9350   
           entity_conf_min        37371.0  0.9843  0.0243  0.901  0.9060   
validation num_embedded_entities  32816.0  2.6360  1.6672  1.000  1.0000   
           entity_conf_mean       32816.0  0.9929  0.0130  0.901  0.9356   
           entity_conf_min        32816.0  0.9846  0.0241  0.901  0.9060   

                                     10%     50%   max  
train      num_embedded_entities  1.0000  2.0000  30.0  
           entity_conf_mean       0.9775  0.9987   1.0  
           entity_conf_min        0.9440  0.9970   1.0  
validation num_embedded_entities  1.0000  2.0000  30.0  
           entity_conf_mean       0.9777  0.9987   1.0  
           entity_conf_min        0.9450  0.9970   1.0

## 5. Ordinary vs confidence-weighted averaging

Two checks on validation:

1. **Vectors:** cosine similarity between each article's two averages.
2. **Ranking:** score each candidate by the cosine similarity between its entity vector and the mean entity vector of the user's history, then compute AUC within each impression (0.5 = random). Labels are only used to evaluate the two versions, never as inputs, and test is not touched.

In [8]:
def unit_rows(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return np.divide(matrix, norms, out=np.zeros_like(matrix), where=norms > 0)


def entity_similarity_scores(events, candidates, news_ids, vectors, chunk=500_000):
    """Cosine similarity between each candidate's entity vector and its user's mean history vector."""
    lookup = pd.Index(news_ids)
    history = events['history_ids'].reset_index(drop=True).explode().dropna()
    history_rows = lookup.get_indexer(history)
    profile_rows = pd.Index(events['event_key']).get_indexer(candidates['event_key'])
    article_rows = lookup.get_indexer(candidates['news_id'])
    assert min(history_rows.min(), profile_rows.min(), article_rows.min()) >= 0, 'Unmatched ID'

    # Summed history vectors point the same way as their mean, which is all cosine needs.
    profiles = np.zeros((len(events), vectors.shape[1]), dtype='float32')
    event_rows = history.index.to_numpy()
    for start in range(0, len(history), chunk):  # Chunks keep memory use low.
        part = slice(start, start + chunk)
        np.add.at(profiles, event_rows[part], vectors[history_rows[part]])

    profiles, articles = unit_rows(profiles), unit_rows(vectors)
    scores = np.empty(len(candidates), dtype='float32')
    for start in range(0, len(candidates), chunk):
        part = slice(start, start + chunk)
        scores[part] = np.einsum('ij,ij->i', profiles[profile_rows[part]], articles[article_rows[part]])
    return scores


def impression_auc(candidates, scores):
    """ROC AUC within each impression (ties count half), via the rank-sum formula."""
    frame = candidates[['event_key', 'label']].assign(score=scores)
    frame['rank'] = frame.groupby('event_key')['score'].rank()
    grouped = frame.groupby('event_key')
    n_pos = grouped['label'].sum().astype('int64')  # int8 would overflow in n_pos * (n_pos + 1).
    n_neg = grouped.size() - n_pos
    pos_rank_sum = frame['rank'].where(frame['label'].eq(1), 0).groupby(frame['event_key']).sum()
    return (pos_rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

In [9]:
events, candidates, news = data['validation']
features, vectors = entity_features_by_partition['validation'], entity_vectors_by_partition['validation']

# Why weighting can only matter a little: confidences sit in a narrow band.
confidences = pd.Series([entity['Confidence'] for field in ENTITY_FIELDS
                         for annotations in news[f'{field}_entities_parsed'] for entity in annotations])
print(f'Entity confidences: {confidences.min():.3f} to {confidences.max():.3f}, '
      f'{confidences.eq(1).mean():.0%} exactly 1.0')

# 1. Vectors: weighting can only change articles with 2+ entities.
multi = features['num_embedded_entities'].ge(2).to_numpy()
similarity = (unit_rows(vectors['ordinary'][multi]) * unit_rows(vectors['weighted'][multi])).sum(axis=1)
print(f'Cosine(ordinary, weighted) over {multi.sum():,} articles with 2+ entities: '
      f'min {similarity.min():.5f}, median {np.median(similarity):.6f}')

# 2. Ranking: mean AUC over validation impressions.
auc = {name: impression_auc(candidates, entity_similarity_scores(events, candidates, news['news_id'], matrix))
       for name, matrix in vectors.items()}
display(pd.DataFrame({'mean impression AUC': {name: values.mean() for name, values in auc.items()}}).round(5))
difference = auc['weighted'] - auc['ordinary']
print(f'Weighted minus ordinary: {difference.mean():+.5f} (95% CI ±{1.96 * difference.sem():.5f}); '
      f'ranking changed in {difference.ne(0).mean():.1%} of impressions')

Entity confidences: 0.901 to 1.000, 62% exactly 1.0
Cosine(ordinary, weighted) over 23,316 articles with 2+ entities: min 0.99732, median 0.999999


,mean impression AUC
ordinary,0.54377
weighted,0.54386


Weighted minus ordinary: +0.00009 (95% CI ±0.00009); ranking changed in 6.6% of impressions


**Result:** Confidence weighting makes no practical difference. Confidences only range from 0.90 to 1.00 (62% are exactly 1.0), so weighting shifts an entity's relative weight by at most ~11%. Even for articles with 2+ entities the two vectors are nearly identical (cosine similarity ≥ 0.997), and validation AUC changes by only +0.0001 (95% CI ±0.0001). Use the ordinary average since it is simpler; both versions are exported for ablations. Entity similarity alone reaches AUC ≈ 0.544, a weak but real signal.

The confidence features are close to 1 for most articles with entities (median mean confidence ≈ 0.999), so their variation comes mostly from the 0s of articles without entities. Treat them as optional and test them in an ablation.

## 6. Export

Under `FEATURES_DIR` (default `outputs/features`), per partition:

- `impression_counts/<version>/<partition>/impression_features.csv`: one row per `event_key`.
- `entity_confidence/<version>/<partition>/entity_features.csv`: one row per `news_id`.
- `entity_confidence/<version>/<partition>/entity_vectors.npz`: `news_id`, `ordinary` and `weighted` arrays (float32, rows aligned with `news_id`).

Re-running overwrites only these files.

In [10]:
for partition in PARTITIONS:
    folder = IMPRESSION_OUTPUT_DIR / partition
    folder.mkdir(parents=True, exist_ok=True)
    impression_features_by_partition[partition].to_csv(folder / 'impression_features.csv', index=False, encoding='utf-8')

    folder = ENTITY_OUTPUT_DIR / partition
    folder.mkdir(parents=True, exist_ok=True)
    features = entity_features_by_partition[partition]
    features.to_csv(folder / 'entity_features.csv', index=False, encoding='utf-8')
    np.savez_compressed(folder / 'entity_vectors.npz', news_id=features['news_id'].to_numpy(dtype=str),
                        **entity_vectors_by_partition[partition])
print('Saved to', IMPRESSION_OUTPUT_DIR, 'and', ENTITY_OUTPUT_DIR)

Saved to /Users/roniheikkonen/Desktop/NUS/Y3S1/BT4222/bt4222/outputs/features/impression_counts/v1 and /Users/roniheikkonen/Desktop/NUS/Y3S1/BT4222/bt4222/outputs/features/entity_confidence/v1


## 7. Using the features

Impression features join to candidate rows on `event_key` and entity features on `news_id` (many candidates per impression/article). The same joins work on `sample_negatives` output.

```python
impression = pd.read_csv(IMPRESSION_OUTPUT_DIR / 'train/impression_features.csv', dtype={'event_key': 'string'})
entity = pd.read_csv(ENTITY_OUTPUT_DIR / 'train/entity_features.csv', dtype={'news_id': 'string'})
examples = (train_candidates
            .merge(impression, on='event_key', how='left', validate='many_to_one')
            .merge(entity, on='news_id', how='left', validate='many_to_one'))

vectors = dict(np.load(ENTITY_OUTPUT_DIR / 'train/entity_vectors.npz'))  # dict(): decompress each array once.
rows =pd.Index(vectors['news_id']).get_indexer(examples['news_id'])
candidate_entity_vectors = vectors['ordinary'][rows]  # All-zero rows: no entity vector.
```

History articles are looked up the same way (each `history_ids` entry), then averaged into a user vector.